# Imports

In [3]:
from moabb.datasets import BNCI2015_013
import mne
from mne.preprocessing import ICA
from mne.decoding import CSP

import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
import pandas as pd
from autoreject import get_rejection_threshold

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, validation_curve, GridSearchCV
from sklearn.metrics import balanced_accuracy_score, make_scorer,confusion_matrix, ConfusionMatrixDisplay, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

In [2]:
dataset = BNCI2015_013()

In [3]:
subjects = [1, 2, 3, 4, 5, 6]
data = dataset.get_data(subjects=subjects)

# Pipeline

In [ ]:
pipe = Pipeline([
    ('csp', CSP(n_components=6)),
    ('scaler', StandardScaler()),
    ('clf', SVC(
        class_weight='balanced',
        kernel='linear',
        C=0.1,
        random_state=42
    ))
])

# Preprocessing

### Dictonaries

In [200]:
event_dict = {'Target': 0, "NonTarget": 1}
X_dict = {}
y_dict = {}    
results = {
    'within': [],
    'pooled': [],
    'cross': []
}

### Loading and preprocessing

In [201]:
for subject_id in subjects:
    print(f"Przetwarzam pacjenta: {subject_id}")
    X_sub_list = []
    y_sub_list = []
    for session_id in data[subject_id].keys():
        for run_id in data[subject_id][session_id].keys():
            raw_run = data[subject_id][session_id][run_id].copy()
            
            # FIR (Bandpass) do wartości 4-12 Hz
            raw_run.filter(l_freq=3.0, h_freq=9.0, fir_design='firwin', verbose=False)

            # CAR - Common Average Reference
            raw_run.set_eeg_reference(verbose=False)
            
            events, _ = mne.events_from_annotations(raw_run, event_id=event_dict, verbose=False)

            epochs = mne.Epochs(
                raw=raw_run,
                events=events,
                event_id=event_dict,
                tmin=-0.5,
                tmax=1.0,
                baseline=(-0.5, 0),
                decim=4,
                preload=True,
                picks='eeg',
                verbose=False,
                reject=None
            )
            #ar = AutoReject(n_jobs=-1, cv=3, verbose=False)

            #reject_params = get_rejection_threshold(epochs, ch_types='eeg', verbose=False)            
            #epochs.drop_bad(reject=reject_params, verbose=False)
            
            epochs.crop(tmin=0.1, tmax=0.5)
            
            X_sub_list.append(epochs.get_data(copy=True))
            y_sub_list.append(epochs.events[:, -1])

    X_dict[subject_id] = np.concatenate(X_sub_list, axis=0)
    y_dict[subject_id] = np.concatenate(y_sub_list, axis=0)

Przetwarzam pacjenta: 1
Przetwarzam pacjenta: 2
Przetwarzam pacjenta: 3
Przetwarzam pacjenta: 4
Przetwarzam pacjenta: 5
Przetwarzam pacjenta: 6


### Fast testing

In [ ]:
sub_id = 1
y_pred = cross_val_predict(pipe, X_dict[sub_id], y_dict[sub_id], cv=skf, verbose=False)
score = balanced_accuracy_score(y_dict[sub_id], y_pred)

print(f"Wynik dla S{sub_id:02d}: {score*100:.2f}%")

# Model

### Within subject

In [202]:
for subject_id in subjects:    
    print(f"Oceniam model dla pacjenta nr: {subject_id}")
    y_pred = cross_val_predict(pipe, X_dict[subject_id], y_dict[subject_id], cv=3)
    for ytrue, ypred in zip(y_dict[subject_id], y_pred):
        results['within'].append({
            'subject': subject_id,
            'y_true': ytrue,
            'y_pred': ypred
        })

Oceniam model dla pacjenta nr: 1
Computing rank from data with rank=None
    Using tolerance 0.00013 (2.2e-16 eps * 64 dim * 9e+09  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 64 dim * 8.6e+09  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with r

### Cross-subject

In [204]:
for test_subject in subjects:
    X_train = np.concatenate([X_dict[subject_id] for s in subjects if s != test_subject], axis=0)
    y_train = np.concatenate([y_dict[subject_id] for s in subjects if s != test_subject], axis=0)
    X_test = X_dict[test_subject]
    
    pipe.fit(X_train, y_train)
    y_pred_cross = pipe.predict(X_test)
    for ytrue, ypred in zip(y_dict[test_subject], y_pred_cross):
        results['cross'].append({
            'subject': test_subject,
            'y_true': ytrue,
            'y_pred': ypred
        })

Computing rank from data with rank=None
    Using tolerance 0.00082 (2.2e-16 eps * 64 dim * 5.8e+10  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 0.00082 (2.2e-16 eps * 64 dim * 5.8e+10  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 0.

### Save to CSV

In [ ]:
pd.DataFrame(results['within']).to_csv("csp_within.csv", index=False)
pd.DataFrame(results['cross']).to_csv("csp_cross.csv", index=False)

# Wizki

In [ ]:
df_within = pd.read_csv("csp_within.csv")
df_cross = pd.read_csv("csp_cross.csv")
results_table = []

### Metrics function

In [207]:
def calculate_metrics(df, experiment_name):

    for subject, group in df.groupby('subject'):
        y_true = group['y_true']
        y_pred = group['y_pred']
        
        acc = balanced_accuracy_score(y_true, y_pred) * 100
        f1 = f1_score(y_true, y_pred, pos_label=1)
        
        results_table.append({
            'Eksperyment': experiment_name,
            'Pacjent': f"S{int(subject):02d}",
            'Balanced Acc (%)': round(acc, 2),
            'F1-Score (Błąd)': round(f1, 3)
        })

In [ ]:
calculate_metrics(df_within, 'Within-Subject')
calculate_metrics(df_cross, 'Cross-Subject')
df_results = pd.DataFrame(results_table)

In [209]:
print("--- TABELA ZBIORCZA WYNIKÓW ---")
print(df_results.to_string(index=False))

--- TABELA ZBIORCZA WYNIKÓW ---
   Eksperyment Pacjent  Balanced Acc (%)  F1-Score (Błąd)
Within-Subject     S01             77.24            0.630
Within-Subject     S02             79.75            0.669
Within-Subject     S03             80.42            0.635
Within-Subject     S04             61.57            0.388
Within-Subject     S05             75.01            0.590
Within-Subject     S06             53.77            0.291
Pooled-Subject     S01             73.50            0.535
Pooled-Subject     S02             76.07            0.597
Pooled-Subject     S03             78.05            0.631
Pooled-Subject     S04             60.62            0.375
Pooled-Subject     S05             69.22            0.485
Pooled-Subject     S06             55.99            0.298
 Cross-Subject     S01             49.45            0.233
 Cross-Subject     S02             49.44            0.323
 Cross-Subject     S03             49.02            0.278
 Cross-Subject     S04             47.40

In [210]:
def plot_confusion_matrices(df, experiment_name, filename):
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
    axes = axes.flatten()
    
    subjects_list = sorted(df['subject'].unique())
    
    for i, sub_id in enumerate(subjects_list):
        ax = axes[i]
        
        sub_data = df[df['subject'] == sub_id]
        y_true = sub_data['y_true']
        y_pred = sub_data['y_pred']
        
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Poprawne', 'BŁĄD'])
        disp.plot(cmap='Blues', ax=ax, values_format='d', colorbar=False)
        
        ax.set_title(f"Pacjent S{int(sub_id):02d}", fontsize=14, fontweight='bold')
        ax.set_xlabel('Przewidziana klasa')
        ax.set_ylabel('Prawdziwa klasa')
    
    plt.suptitle(f"Macierze Pomyłek - {experiment_name}", fontsize=20, y=1.03)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [211]:
plot_confusion_matrices(df_within, 'Within-Subject', 'CSP_Within_Subject.png')
plot_confusion_matrices(df_pooled, 'Pooled Dataset', 'CSP_Pooled.png')
plot_confusion_matrices(df_cross, 'Cross-Subject', 'CSP_Cross_Subject.png')